# Module 21: Vector Database HNSW Index Milvus — Interactive Laboratory

Every cell below runs the module's **real** implementation from
`project_solution/vector_database_engine.py`. Nothing here prints a claim it has not verified.

What you will do:

1. Load the engine and inspect what it actually exports.
2. Run its primary workflow and check the assertions that define correctness.
3. **Commit to a prediction**, then run the cell that tests it.
4. Measure a property rather than asserting one.
5. Fix a deliberately broken cell in place.

> The code in cells 4, 6 and 8 is lifted from this module's own test suite, so it
> cannot drift from the implementation. If the API changes, those tests fail
> first and this notebook is regenerated from them.


## 1. Load the engine and introspect it

Rather than trusting a hardcoded list of class names, ask the module what it
actually contains.


In [ ]:
import inspect
import sys
from pathlib import Path

sys.path.insert(0, str(Path('.').resolve() / 'project_solution'))
import vector_database_engine

classes = [n for n, o in inspect.getmembers(vector_database_engine, inspect.isclass)
           if o.__module__ == 'vector_database_engine']
functions = [n for n, o in inspect.getmembers(vector_database_engine, inspect.isfunction)
             if o.__module__ == 'vector_database_engine']

print('module   : vector_database_engine')
print(f'classes  : {classes}')
print(f'functions: {functions}')
print()
for name in classes:
    obj = getattr(vector_database_engine, name)
    try:
        sig = inspect.signature(obj.__init__)
        params = [p for p in sig.parameters if p != 'self']
    except (TypeError, ValueError):
        params = ['<builtin>']
    print(f'  {name}({", ".join(params)})')

## 2. Baseline: Distance metrics

This is the module's own `test_distance_metrics` — real instantiation, real calls, real
assertions. If it runs clean, the property it encodes holds.


In [ ]:
import pytest
from vector_database_engine import (
    HNSWIndex,
    ScalarQuantizer8,
    cosine_distance,
    euclidean_distance,
)

v1 = [1.0, 0.0, 0.0]
v2 = [1.0, 0.0, 0.0]
v3 = [0.0, 1.0, 0.0]

# Identical vectors: distance should be 0.0
assert cosine_distance(v1, v2) == pytest.approx(0.0, abs=1e-5)
assert euclidean_distance(v1, v2) == pytest.approx(0.0, abs=1e-5)

# Orthogonal vectors: cosine distance is 1.0 (since similarity is 0.0)
assert cosine_distance(v1, v3) == pytest.approx(1.0, abs=1e-5)
assert euclidean_distance(v1, v3) == pytest.approx(1.41421356, abs=1e-4)

# Dimension mismatch
with pytest.raises(ValueError):
    cosine_distance([1.0], [1.0, 2.0])

print('PASSED: test_distance_metrics')

## 3. 🔮 Prediction — commit before you run

An HNSW index returns 10 nearest neighbours. Predict whether they are guaranteed to be the true 10 nearest, and what knob trades recall for latency.

Write your answer down. An uncommitted guess teaches nothing, because you will
retro-fit it to whatever the next cell prints.

The next cell runs `test_scalar_quantization_sq8`, which tests exactly this property.


In [ ]:
vec = [0.1, -0.5, 1.2, 3.4, -2.1]
qv = ScalarQuantizer8.quantize(vec)

assert len(qv.data) == len(vec)
assert qv.dim == len(vec)

# Dequantized reconstruction should have low error
reconstructed = ScalarQuantizer8.dequantize(qv)
for orig, rec in zip(vec, reconstructed, strict=False):
    assert abs(orig - rec) < 0.05  # Within 8-bit resolution error

print('PASSED: test_scalar_quantization_sq8')

## 4. Measure it: Hnsw insertion and graph properties

An assertion tells you a property holds. A measurement tells you *how much*.
This cell runs `test_hnsw_insertion_and_graph_properties` and times it.


In [ ]:
import time

_t0 = time.perf_counter()

dim = 8
hnsw = HNSWIndex(dim=dim, metric="euclidean", M=4, random_seed=42)

for i in range(25):
    vec = [float(i + j) for j in range(dim)]
    hnsw.insert(f"doc_{i}", vec, metadata={"idx": i})

assert hnsw.entry_point is not None
assert hnsw.max_level >= 0
assert len(hnsw.nodes) == 25

# Check maximum degree constraint across layers
for lvl in range(1, hnsw.max_level + 1):
    for _node_id, neighbors in hnsw.layers[lvl].items():
        # Outgoing links should be bounded by M
        assert len(neighbors) <= hnsw.M + 1  # Allowing small boundary margin during edge linking

_elapsed = (time.perf_counter() - _t0) * 1000
print('PASSED: test_hnsw_insertion_and_graph_properties')
print(f'wall clock: {_elapsed:.2f} ms')

## 5. 🛠️ Fix this cell — it is deliberately broken

The cell below asserts something **false** about the real object. Read the
failure, work out the true value from the module's actual behaviour, and correct
the expected number.

Do not delete the assertion. The point is to make it pass by knowing the answer.


In [ ]:
# DELIBERATELY BROKEN - fix the expected value below.
# Hint: print the real value first, then decide what the assertion should say.

exports = [n for n in dir(vector_database_engine) if not n.startswith('_')]
print(f'actual export count: {len(exports)}')
print(f'actual exports     : {exports}')

EXPECTED_EXPORT_COUNT = 999      # <-- wrong on purpose. Replace it.

assert len(exports) == EXPECTED_EXPORT_COUNT, (
    f'expected {EXPECTED_EXPORT_COUNT} exports, found {len(exports)}. '
    'Read the printed value above and correct the constant.'
)
print('Fixed - assertion now reflects reality.')

### 🎓 Key takeaways

1. ANN indexes trade recall for latency - and recall must be measured, not assumed.
2. HNSW's layered graph is what turns a linear scan into a logarithmic walk.
3. Filtered vector search is a different problem from pure similarity search.

---

**Continue with this module:**

- [README.md](README.md) — the mental model and failure modes
- [PROJECT_GUIDE.md](PROJECT_GUIDE.md) — build it yourself, in 3 tiers
- [starter/](starter/) — your stubs; run the tests from there to grade yourself
- [debug_lab/SYMPTOMS.md](debug_lab/SYMPTOMS.md) — diagnose planted bugs from the symptom
- [TROUBLESHOOTING_AND_EDGE_CASES.md](TROUBLESHOOTING_AND_EDGE_CASES.md) — real errors, real causes
- [SELF_ASSESSMENT_AND_CHALLENGES.md](SELF_ASSESSMENT_AND_CHALLENGES.md) — quiz and diagnostics
